# RAG Extension: Retrieval-Augmented Microservice Boundary Agent (Lab 1.6)

**System description.** The AI agent from Lab 1.5 receives a Django module (X: structural features and dependencies)
and returns the microservice cluster it belongs to (y). In this notebook the agent is extended with **retrieval**:
for every new X the three most similar known (X, y) examples are found in a **ChromaDB** vector database and
inserted into the prompt sent to Gemini.

**Pipeline:** `input X` -> `retrieve 3 similar examples (ChromaDB)` -> `augmented prompt` -> `Gemini` -> `output y`

**Data note.** The 1,000+ examples are generated by rules (seeded, reproducible) from typical Django app profiles of an
online school. They are illustrative, not measurements of the real system. The real extracted data would replace them.

In [1]:
# Install dependencies (ChromaDB = vector database, google-genai = Gemini client)
!pip install -q -U chromadb google-genai pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


## Setup: Gemini client
The API key is stored in **Colab -> Secrets** under the name `GEMINI_KEY` (never hard-coded).

In [2]:
import os, re, time, random
import pandas as pd
from google import genai
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI_KEY')
client = genai.Client(api_key=GEMINI_KEY)
MODEL = 'gemini-3.5-flash'   # the model that worked in Lab 1.5

def ask(prompt, retries=4):
    """Call Gemini; retry on temporary errors such as rate limits."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt).text
        except Exception as e:
            print(f'  retry {attempt + 1}: {str(e)[:90]}')
            time.sleep(10 * (attempt + 1))
    return ''
print('Gemini client ready, model:', MODEL)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Gemini client ready, model: gemini-3.5-flash


## Stage 1. Data: generate 1,000+ (X, y) examples
Each example is one Django app. **X** = module name, feature vector (imports, foreign keys, Celery tasks, DRF serializers)
and typed dependencies (code / ORM / framework). **y** = microservice cluster.
Six clusters have different feature profiles, as in the thesis (e.g. billing apps have many foreign keys, notification apps many Celery tasks).
Two examples are the exact known pairs from Lab 1.2 / 1.4 (`payments`, `homework_balancer`).

In [3]:
random.seed(42)   # reproducible dataset
CHANNELS = ['Code', 'ORM', 'Framework']

# cluster -> typical app names and feature ranges (min, max)
CLUSTERS = {
    'Financial/Billing Service': dict(
        stems=['payments', 'invoices', 'billing', 'fiscalization', 'refunds', 'subscriptions', 'tariffs', 'receipts', 'acquiring', 'promo_codes'],
        imports=(8, 16), fks=(15, 35), celery=(4, 10), drf=(8, 18)),
    'Educational Core Service': dict(
        stems=['homework_balancer', 'homework_iterations', 'courses', 'lessons', 'quizzes', 'curriculum', 'grading', 'webinars', 'streams', 'attendance'],
        imports=(6, 12), fks=(8, 20), celery=(1, 4), drf=(6, 14)),
    'Notification & Communication Service': dict(
        stems=['notifications', 'email_templates', 'sms_gateway', 'push_messages', 'chat_messages', 'mailing', 'telegram_bot', 'reminders', 'announcements', 'digests'],
        imports=(10, 18), fks=(2, 8), celery=(8, 15), drf=(2, 6)),
    'Gamification & Engagement Service': dict(
        stems=['rewards', 'badges', 'leaderboards', 'achievements', 'streaks', 'points', 'quests', 'levels', 'challenges', 'xp_events'],
        imports=(6, 11), fks=(6, 14), celery=(3, 7), drf=(4, 8)),
    'User & Identity Service': dict(
        stems=['users', 'auth', 'profiles', 'roles', 'sessions', 'student_cards', 'teachers', 'parents', 'permissions', 'invitations'],
        imports=(8, 14), fks=(10, 25), celery=(0, 3), drf=(8, 16)),
    'Shared Infrastructure Service': dict(
        stems=['app_settings', 'logging', 'feature_flags', 'file_storage', 'audit', 'admin_panel', 'amocrm_sync', 'analytics_export', 'search_index', 'healthchecks'],
        imports=(10, 20), fks=(1, 8), celery=(2, 9), drf=(0, 5)),
}
SUFFIXES = ['', '_api', '_core', '_admin', '_history', '_events', '_tasks', '_v2', '_legacy',
            '_reports', '_hooks', '_sync', '_export', '_import', '_models', '_services', '_utils', '_signals']

def make_deps(stem, cluster):
    """1-2 peers from the same cluster plus common cross-cluster links, each with a set of channels."""
    peers = [s for s in CLUSTERS[cluster]['stems'] if s != stem]
    deps = [(p, random.sample(CHANNELS, random.randint(1, 3))) for p in random.sample(peers, 2)]
    if cluster != 'User & Identity Service' and random.random() < 0.6:
        deps.append(('users', ['ORM']))
    if cluster != 'Shared Infrastructure Service' and random.random() < 0.3:
        deps.append(('app_settings', ['Code']))
    return '; '.join(f"{n} ({', '.join(c)})" for n, c in deps)

def x_text(name, imports, fks, celery, drf, deps):
    return (f"Module `{name}`. Features: [Imports: {imports}, FKs: {fks}, "
            f"Celery Tasks: {celery}, DRF Serializers: {drf}]. Dependencies: {deps}")

rows = []
for y, cfg in CLUSTERS.items():
    for stem in cfg['stems']:
        for suf in SUFFIXES:
            name = stem + suf
            f = (random.randint(*cfg['imports']), random.randint(*cfg['fks']),
                 random.randint(*cfg['celery']), random.randint(*cfg['drf']))
            rows.append(dict(name=name, imports=f[0], fks=f[1], celery=f[2], drf=f[3],
                             deps=make_deps(stem, y), y=y))

df = pd.DataFrame(rows)
# the two known pairs from Lab 1.2 / 1.4, with their exact features
df.loc[df.name == 'payments', ['imports', 'fks', 'celery', 'drf']] = [12, 29, 8, 15]
df.loc[df.name == 'homework_balancer', ['imports', 'fks', 'celery', 'drf']] = [8, 14, 2, 10]
df['x'] = [x_text(r.name, r.imports, r.fks, r.celery, r.drf, r.deps) for r in df.itertuples()]

# hold out 20 examples to test the system on unseen modules
protected = ['payments', 'homework_balancer']
test_df = df[~df.name.isin(protected)].sample(20, random_state=1)
db_df = df.drop(test_df.index).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

df.to_csv('examples.csv', index=False)
print(f'Generated {len(df)} examples ({len(db_df)} for the database, {len(test_df)} held out)')
print(df.y.value_counts().to_string())
df[['name', 'x', 'y']].head(3)

Generated 1080 examples (1060 for the database, 20 held out)
y
Financial/Billing Service               180
Educational Core Service                180
Notification & Communication Service    180
Gamification & Engagement Service       180
User & Identity Service                 180
Shared Infrastructure Service           180


,name,x,y
0,payments,"Module `payments`. Features: [Imports: 12, FKs...",Financial/Billing Service
1,payments_api,"Module `payments_api`. Features: [Imports: 14,...",Financial/Billing Service
2,payments_core,"Module `payments_core`. Features: [Imports: 8,...",Financial/Billing Service


## Stage 2. Store all examples in ChromaDB
Each example is stored as a text document (the description of X) with its label `y` as metadata.
ChromaDB turns every document into an **embedding** (default model: all-MiniLM-L6-v2) for semantic similarity search.

In [4]:
import chromadb

chroma = chromadb.Client()   # in-memory database
collection = chroma.get_or_create_collection(name='django_modules', metadata={'hnsw:space': 'cosine'})

BATCH = 200
for start in range(0, len(db_df), BATCH):
    part = db_df.iloc[start:start + BATCH]
    collection.add(ids=[f'ex{i}' for i in part.index],
                   documents=part.x.tolist(),
                   metadatas=[{'name': n, 'y': y} for n, y in zip(part.name, part.y)])
print('Examples in ChromaDB:', collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 43.4MiB/s]


Examples in ChromaDB: 1060


## Stage 3. Retrieval: find the 3 most similar examples for a new X

In [5]:
def retrieve(x, k=3):
    """Return the k most similar stored examples for the description x."""
    res = collection.query(query_texts=[x], n_results=k)
    return [dict(x=d, y=m['y']) for d, m in zip(res['documents'][0], res['metadatas'][0])]

# A NEW module that is not in the database (same input as the few-shot prompt in Lab 1.4)
new_x = x_text('gamification_rewards', 9, 11, 5, 6, 'homework_balancer (Code); users (ORM)')
print('New X:\n', new_x, '\n')
examples = retrieve(new_x)
for i, e in enumerate(examples, 1):
    print(f'Retrieved {i}: {e["x"]}\n   y = {e["y"]}\n')

New X:
 Module `gamification_rewards`. Features: [Imports: 9, FKs: 11, Celery Tasks: 5, DRF Serializers: 6]. Dependencies: homework_balancer (Code); users (ORM) 

Retrieved 1: Module `homework_balancer_import`. Features: [Imports: 12, FKs: 9, Celery Tasks: 3, DRF Serializers: 12]. Dependencies: grading (ORM, Framework, Code); homework_iterations (ORM); users (ORM)
   y = Educational Core Service

Retrieved 2: Module `homework_balancer_utils`. Features: [Imports: 9, FKs: 15, Celery Tasks: 4, DRF Serializers: 7]. Dependencies: homework_iterations (Framework, Code); courses (Framework, ORM, Code); users (ORM)
   y = Educational Core Service

Retrieved 3: Module `homework_balancer`. Features: [Imports: 8, FKs: 14, Celery Tasks: 2, DRF Serializers: 10]. Dependencies: homework_iterations (Framework); courses (Framework, Code, ORM); users (ORM)
   y = Educational Core Service



## Stage 4. Augmented prompt
The retrieved examples are inserted into the prompt as known (X, y) mappings (few-shot), but chosen automatically instead of by hand.

In [6]:
LABELS = list(CLUSTERS.keys())
INSTRUCTIONS = ('You are an AI Architect Agent specializing in decomposing Django monoliths into microservices.\n'
                'Assign the Django module to exactly one of these microservice clusters:\n'
                + '\n'.join(f'- {l}' for l in LABELS) + '\n')
ANSWER_RULE = ('\nExplain your reasoning briefly, then end with one last line in exactly this form:\n'
               'FINAL ANSWER: <cluster name>')

def zero_shot_prompt(x):
    return INSTRUCTIONS + f'\nInput X:\n{x}\n' + ANSWER_RULE

def rag_prompt(x, examples):
    shots = '\n'.join(f'Example {i}:\nInput X: {e["x"]}\nOutput y: "{e["y"]}"\n' for i, e in enumerate(examples, 1))
    return (INSTRUCTIONS + '\nHere are the most similar known mappings retrieved from the database:\n' + shots +
            f'\nNow process the new input:\nInput X:\n{x}\n' + ANSWER_RULE)

print(rag_prompt(new_x, examples))

You are an AI Architect Agent specializing in decomposing Django monoliths into microservices.
Assign the Django module to exactly one of these microservice clusters:
- Financial/Billing Service
- Educational Core Service
- Notification & Communication Service
- Gamification & Engagement Service
- User & Identity Service
- Shared Infrastructure Service

Here are the most similar known mappings retrieved from the database:
Example 1:
Input X: Module `homework_balancer_import`. Features: [Imports: 12, FKs: 9, Celery Tasks: 3, DRF Serializers: 12]. Dependencies: grading (ORM, Framework, Code); homework_iterations (ORM); users (ORM)
Output y: "Educational Core Service"

Example 2:
Input X: Module `homework_balancer_utils`. Features: [Imports: 9, FKs: 15, Celery Tasks: 4, DRF Serializers: 7]. Dependencies: homework_iterations (Framework, Code); courses (Framework, ORM, Code); users (ORM)
Output y: "Educational Core Service"

Example 3:
Input X: Module `homework_balancer`. Features: [Imports

## Stage 5. Generation: send the augmented prompt to Gemini

In [7]:
def final_answer(text):
    m = re.findall(r'FINAL ANSWER:\s*(.+)', text or '')
    return m[-1].strip().strip('*`" ') if m else ''

answer = ask(rag_prompt(new_x, examples))
print(answer)
print('\n=== y (predicted cluster):', final_answer(answer))

Reasoning:
The module `gamification_rewards` directly addresses engagement mechanics such as user rewards, achievements, and points integration. Although it interacts with `homework_balancer` and `users`, its core business domain logic aligns directly with rewarding user actions and driving engagement.

FINAL ANSWER: Gamification & Engagement Service

=== y (predicted cluster): Gamification & Engagement Service


## Stage 6. Does retrieval help? Zero-shot vs RAG on held-out modules
Both variants see the same list of allowed clusters; RAG additionally sees the 3 retrieved examples.
The test uses 10 unseen modules (kept small to stay inside the free API limits).

In [8]:
N_TEST = 10
results = []
for r in test_df.head(N_TEST).itertuples():
    zs = final_answer(ask(zero_shot_prompt(r.x)));  time.sleep(4)
    rg = final_answer(ask(rag_prompt(r.x, retrieve(r.x))));  time.sleep(4)
    results.append(dict(module=r.name, true_y=r.y, zero_shot=zs, rag=rg,
                        zs_ok=zs == r.y, rag_ok=rg == r.y))
res = pd.DataFrame(results)
print(f'Zero-shot accuracy: {res.zs_ok.mean():.0%}   RAG accuracy: {res.rag_ok.mean():.0%}   (n={len(res)})')
res

  retry 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing 
  retry 2: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing 
  retry 1: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quot
  retry 1: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quot
Zero-shot accuracy: 70%   RAG accuracy: 100%   (n=10)


,module,true_y,zero_shot,rag,zs_ok,rag_ok
0,chat_messages_events,Notification & Communication Service,Notification & Communication Service,Notification & Communication Service,True,True
1,attendance,Educational Core Service,Educational Core Service,Educational Core Service,True,True
2,rewards_legacy,Gamification & Engagement Service,Gamification & Engagement Service,Gamification & Engagement Service,True,True
3,subscriptions_api,Financial/Billing Service,Financial/Billing Service,Financial/Billing Service,True,True
4,telegram_bot_tasks,Notification & Communication Service,Notification & Communication Service,Notification & Communication Service,True,True
5,chat_messages_reports,Notification & Communication Service,Notification & Communication Service,Notification & Communication Service,True,True
6,amocrm_sync_import,Shared Infrastructure Service,User & Identity Service,Shared Infrastructure Service,False,True
7,amocrm_sync_signals,Shared Infrastructure Service,Notification & Communication Service,Shared Infrastructure Service,False,True
8,lessons_hooks,Educational Core Service,Educational Core Service,Educational Core Service,True,True
9,student_cards_export,User & Identity Service,Educational Core Service,User & Identity Service,False,True


## Reflection
*This notebook extends the end-to-end agent from Lab 1.5 with retrieval. For a new Django module (X: name, imports, foreign keys, Celery tasks, DRF serializers and typed dependencies) the system retrieves the three most similar examples from a ChromaDB database of 1,060 generated (X, y) pairs, inserts them into the prompt and asks Gemini (gemini-3.5-flash) for the microservice cluster y. Compared with Lab 1.5, the few-shot examples are no longer chosen by hand but found automatically in a database of more than 1,000 examples. On 10 held-out modules, zero-shot prompting reached 70% accuracy and RAG reached 100%. The three zero-shot mistakes were modules with ambiguous names (amocrm_sync_*, student_cards_export); the retrieved neighbours showed the model how similar modules are labelled. The results need care: the data are generated by rules, only 10 modules were tested, and the test modules share name stems with stored ones, which favours retrieval. Free-tier API limits (503 and 429 errors) required automatic retries. Next steps are to use real data extracted from the Django project, request JSON output and compare the predictions with the expert decomposition.